In [ ]:
%load_ext autoreload
%autoreload 2
# %matplotlib widget
%pdb off

from matplotlib import pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import plotly
from IPython.display import display, HTML
import flowEmulationUtils as feUtils
import random
from scipy.stats import lognorm

plotly.offline.init_notebook_mode()
display(HTML(
    '<script type="text/javascript" async src="https://cdnjs.cloudflare.com/ajax/libs/mathjax/2.7.1/MathJax.js?config=TeX-MML-AM_SVG"></script>'
))

#close all figures
plt.close('all')
plt.rcParams['figure.dpi'] = 140
im_scaling = .75
plt.rcParams['figure.figsize'] = [6.4 * im_scaling, 4.8 * im_scaling]

home_dir = "./"
display(home_dir)
plt.close('all')

# Load Data

In [ ]:
multiRun_dir = f"{home_dir}/CHARLES/multiRuns/"
plotFolder = f"{multiRun_dir}"

roomVentilationMI = pd.read_csv(f"{multiRun_dir}/roomVentilationMIEmulation.csv", index_col = [0,1])
flowStatsMI = pd.read_csv(f"{multiRun_dir}/flowStatsMIEmulation.csv", index_col=[0,1])

In [ ]:
# Train/Dev/Test Assignment
# Create a column called split and assign 70 % to train, 10% to dev, and 20% to test in roomVentilationMI
random.seed(42)  # For reproducibility
roomVentilationMI["split"] = roomVentilationMI.index.to_series().apply(lambda _: random.random())
roomVentilationMI["split"] = roomVentilationMI["split"].apply(lambda x: "train" if x < 0.7 else ("dev" if x < 0.8 else "test"))

for (run, room), row in roomVentilationMI.iterrows():
    windowKeyCols = roomVentilationMI.columns[
        roomVentilationMI.columns.str.contains("windowKeys")
    ].tolist()
    windowKeys = row[windowKeyCols].dropna()
    for windowKey in windowKeys:
        flowStatsMI.loc[(run, windowKey), "split"] = row["split"]

In [ ]:
df = flowStatsMI.copy()
# df = df[df["slAll"] == False]
# normalize x_cols. Flow quantities to be normalized by WS. Pressures to be normalized by W**2:

# df['p-noInt_optp0-q_modelC_d'] = df['p-noInt_optp0-q_model'] * df['p-noInt_optp0-C_d']
# for col in df.columns:
#     if "mag" in col or "shear" in col or "normal" in col:
#         df[col] = df[col] / df['p-noInt_optp0-q_modelC_d']

df["skylight"] = df['openingType'].apply(lambda x: 1 if "skylight" in x else 0)
df["cross"] = df['openingType'].apply(lambda x: 1 if "cross" in x else 0)
df["single"] = df['openingType'].apply(lambda x: 1 if "single" in x else 0)
df["dual"] = df['openingType'].apply(lambda x: 1 if "dual" in x else 0)
df["corner"] = df['openingType'].apply(lambda x: 1 if "corner" in x else 0)
Sdelp = np.sign(df['p-noInt_optp0-q_model'])
Sdelp[Sdelp == 0] = 1  # Assign 1 to zero values
df["Sdelp"] = Sdelp
df["EP_shear-noInt-qIn"] = df["EP_shear-noInt"] * df["Sdelp"] > 0
df["EP_shear-noInt-qOut"] = df["EP_shear-noInt"] * df["Sdelp"] <= 0
df["EP_shear_o_qmodel"] = df["EP_shear-noInt"] / df["p-noInt_optp0-q_model"]
df["rev-mass_flux"] =df["net-mass_flux"] -  df["mean-mass_flux"].abs()
df["rev-mass_flux-Norm"] = df["rev-mass_flux"] / df["WS"]
df["p_intensity-noInt"] = df["p_rms-noInt"] / df["p-noInt_optp0-q_model"]**2

p_norm_cols = []
u_norm_cols = []
no_norm_cols = []
for col in df.columns:
    if ("noInt" in col or col == "flux" or "EP" in col) and "Norm" not in col:
        if "-p0" in col or "p_" in col or "(p)" in col or "p0meas" in col or "u**2" in col:
            p_norm_cols.append(col)
        elif "mag" in col or "shear" in col or "normal" in col or "flux" in col or "(u" in col or "q_model" in col:
            u_norm_cols.append(col)
        else:
            no_norm_cols.append(col)

print(f"Normalizing p cols: {sorted(p_norm_cols)}")
print(f"Normalizing u cols: {sorted(u_norm_cols)}")
print(f"Not normalizing cols: {sorted(no_norm_cols)}")

# Normalize pressure columns by WS^2
for col in p_norm_cols:
    df[f"{col}-Norm"] = df[col].div(df["WS"]**2, axis=0)
# Normalize velocity columns by WS
for col in u_norm_cols:
    df[f"{col}-Norm"] = df[col].div(df["WS"], axis=0)

In [ ]:
grouped = df.groupby("WS").mean(numeric_only=True)
ratio = grouped.loc[4] / grouped.loc[2]  # Should be about 4x difference`
ratio = pd.DataFrame(ratio).reset_index()
ratio

In [ ]:
for col in df.columns:
    if "orientation" in col:
        sin_col = col.replace("orientation", "sin")
        cos_col = col.replace("orientation", "cos")
        df[sin_col] = np.sin(df[col])
        df[cos_col] = np.cos(df[col])

In [ ]:
df = df[~((df["roomType"] == "single") & (df["slAll"] == False))]  # remove single rooms without skylights
roomVentilationMI = roomVentilationMI[~((roomVentilationMI["roomType"] == "single") & (roomVentilationMI["slAll"] == False))]  # remove single rooms without skylights

df["all"] = True

In [ ]:
y_var = "flux-Norm"
x_var = "p-noInt_optp0-q_model-Norm"
# x_var = "EP_shear_o_qmodel"
ydf = df[y_var].copy()

x_cols = [x_var, "p_intensity-noInt", "rms-mass_flux-Norm", "p_rms-noInt-Norm", "EP_normal-noInt-Norm", "EP_shear-noInt-Norm", "EP_vel_orientation-noInt", "rev-mass_flux-Norm"]#, "p-noInt_optp0-p0", "p_avg-noInt"]#, "Sdelp", "skylight", "delT", "SS", "EP_vel_sin", "EP_vel_cos", "EP_vel_orientation"]#, "EP_shear-noInt-qIn","EP_shear-noInt-qOut"]

x_cols += ["split"]
xdf = df[x_cols].copy()

display(xdf.columns.values)

# ydf = np.log(ydf + .0001)
# x_df = np.log(xdf + .0001)

In [ ]:
# xdf = xdf.map(lambda s: abs(s) if isinstance(s, (int, float)) else s)  # Ensure all numeric values are positive
# ydf = np.abs(ydf)

In [ ]:
x_data = {}
y_data = {}
# first_level_col = "roomType"
first_level_col = "Sdelp"  # Use Sdelp to group data by sign of p-noInt_optp0-q_model
second_level_col = "skylight"
for first_level in df[first_level_col].unique():
    for second_level in df[second_level_col].unique():
        rows = (df[first_level_col] == first_level) & (df[second_level_col] == second_level)
        if rows.sum() > 0:
            x_data[(first_level, second_level)] = xdf[rows]
            y_data[(first_level, second_level)] = ydf[rows]


In [ ]:
def recombine_grouped_data(grouped_data):
    df = pd.DataFrame()
    for (first_level, second_level), data in grouped_data.items():
        if df.empty:
            df = data.copy()
        else:
            df = pd.concat([df, data], axis=0)
    return df

In [ ]:
def calculate_linear_likelihood(y, y_pred):
    """
    Calculate log likelihood for a linear regression model.
    
    Parameters:
    -----------
    X : array-like, shape (n_samples, n_features)
        Input features
    y : array-like, shape (n_samples,)
        Target values
        
    Returns:
    --------
    log_likelihood : float
        Log likelihood of the data under the model
    """
    import numpy as np
    from scipy import stats
    
    # Calculate residuals
    residuals = y - y_pred
    
    # Estimate variance (MLE of variance)
    n = len(y)
    variance = np.sum(residuals**2) / n
    
    # Calculate average log likelihood
    log_likelihood = np.mean(stats.norm.logpdf(y, loc=y_pred, scale=np.sqrt(variance)))
    
    return log_likelihood

def calculate_normalized_rmse(y, y_pred, normalization='std'):
    """
    Calculate RMSE and normalized RMSE for any regression model.
    
    Parameters:
    -----------
    X : array-like, shape (n_samples, n_features)
        Input features
    y : array-like, shape (n_samples,)
        Target values
    normalization : str, optional (default='std')
        Method for normalization:
        - 'std': normalize by standard deviation of y
        - 'mean': normalize by mean of y
        - 'range': normalize by range of y
        
    Returns:
    --------
    rmse : float
        Root Mean Square Error
    nrmse : float
        Normalized Root Mean Square Error
    """
    import numpy as np
    
    # Calculate RMSE
    mse = np.mean((y - y_pred)**2)
    rmse = np.sqrt(mse)
    
    # Calculate normalized RMSE
    if normalization == 'mean':
        # Normalize by mean of observed values
        nrmse = rmse / np.mean(np.abs(y))
    elif normalization == 'range':
        # Normalize by range of observed values
        nrmse = rmse / (np.max(y) - np.min(y))
    else:  # default: 'std'
        # Normalize by standard deviation of observed values
        nrmse = rmse / np.std(y)
    
    return rmse, nrmse

def visualize_linear_model(linear_model, X, y, y_pred=None, hue=None, style=None, model_name="Linear Regression", feature_names=None, top_features=10, y_transformer=None):
    """
    Create a comprehensive visualization of linear model fit with feature importance.
    
    Parameters:
    -----------
    linear_model : sklearn linear model (LinearRegression, Ridge, Lasso, etc.)
        The fitted linear model
    X : DataFrame
        Input features as pandas DataFrame
    y : Series or array
        Target values
    model_name : str, optional
        Name of the model type for plot titles
    feature_names : list, optional
        Names of features (if not provided, will use X.columns)
    top_features : int, optional
        Number of top features to show in importance plot
    """
    import numpy as np
    import matplotlib.pyplot as plt
    import pandas as pd
    from sklearn.metrics import mean_squared_error, r2_score
    import seaborn as sns
    
    # Convert y to numpy array if it's a pandas Series
    if hasattr(y, 'values'):
        y_values = y.values
    else:
        y_values = np.array(y)

    if hue is not None and hasattr(X, 'index'):
        hue = hue.loc[X.index]
    if style is not None and hasattr(X, 'index'):
        style = style.loc[X.index]
    
    # Get feature names from DataFrame if not provided
    if feature_names is None and hasattr(X, 'columns'):
        feature_names = X.columns.tolist()
    elif feature_names is None:
        feature_names = [f'Feature {i}' for i in range(X.shape[1])]
    
    # Get predictions
    if y_pred is None:
        y_pred = linear_model.predict(X)

    if y_transformer is not None:
        # Inverse transform predictions if a transformer is provided
        y_pred = y_transformer.inverse_transform(y_pred.reshape(-1, 1)).ravel()
        y_values = y_transformer.inverse_transform(y_values.reshape(-1, 1)).ravel()
    # Calculate metrics
    r2 = r2_score(y_values, y_pred)
    rmse, nrmse = calculate_normalized_rmse(y_values, y_pred)
    log_likelihood = calculate_linear_likelihood(y_values, y_pred)
    
    # Calculate residuals
    residuals = y_values - y_pred
    
    # Create figure with subplots
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # 1. Actual vs Prediction plot (swapped axes)
    sns.scatterplot(x=y_pred, y=y_values, hue=hue, style=style, alpha=0.6, ax=axes[0, 0])
    
    # Add perfect prediction line
    min_val = min(min(y_values), min(y_pred))
    max_val = max(max(y_values), max(y_pred))
    axes[0, 0].plot([min_val, max_val], [min_val, max_val], 'r--')
    
    axes[0, 0].set_xlabel('Predicted Values')
    axes[0, 0].set_ylabel('Actual Values')
    axes[0, 0].set_title(f'{model_name}: Actual vs Prediction\nR² = {r2:.4f}, NRMSE = {nrmse:.4f}, RMSE = {rmse:.4f}, Log Likelihood = {log_likelihood:.4f}')
    axes[0, 0].grid(True, alpha=0.3)
    
    # 2. Residuals vs Actual
    sns.scatterplot(x=y_values, y=residuals, hue=hue, style=style, alpha=0.6, ax=axes[0, 1])
    axes[0, 1].axhline(y=0, color='r', linestyle='--')
    axes[0, 1].set_xlabel('Actual Values')
    axes[0, 1].set_ylabel('Residuals')
    axes[0, 1].set_title('Residuals vs Actual')
    axes[0, 1].grid(True, alpha=0.3)
    
    # 3. Residual Distribution
    sns.histplot(residuals, kde=True, ax=axes[1, 0])
    axes[1, 0].axvline(x=0, color='r', linestyle='--')
    axes[1, 0].set_xlabel('Residual Value')
    axes[1, 0].set_ylabel('Frequency')
    axes[1, 0].set_title('Residual Distribution')
    axes[1, 0].grid(True, alpha=0.3)
    
    # 4. QQ Plot for Residuals
    from scipy import stats
    stats.probplot(residuals, dist="norm", plot=axes[1, 1])
    axes[1, 1].set_title('Q-Q Plot of Residuals')
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Feature Importance from Coefficients
    if hasattr(linear_model, 'coef_'):
        # Get coefficients
        if linear_model.coef_.ndim > 1:
            coeffs = linear_model.coef_[0]  # For multi-output models
        else:
            coeffs = linear_model.coef_
        
        # Create DataFrame with features and coefficients
        coef_df = pd.DataFrame({
            'Feature': feature_names,
            'Coefficient': coeffs
        })
        
        # Add absolute coefficient for ranking
        coef_df['Abs_Coefficient'] = np.abs(coef_df['Coefficient'])
        
        # Sort by absolute coefficient
        coef_df = coef_df.sort_values('Abs_Coefficient', ascending=False)
        
        # Show top features
        top_coef = coef_df.head(top_features)
        
        # Plot coefficients
        plt.figure(figsize=(12, 8))
        colors = ['red' if x < 0 else 'blue' for x in top_coef['Coefficient']]
        bars = sns.barplot(x='Coefficient', y='Feature', data=top_coef, palette=colors)
        
        # Add a vertical line at x=0
        plt.axvline(x=0, color='black', linestyle='-', alpha=0.3)
        
        # Add value labels to the bars
        for i, v in enumerate(top_coef['Coefficient']):
            bars.text(v + (0.01 if v >= 0 else -0.01), i, f"{v:.4f}", 
                     va='center', ha='left' if v >= 0 else 'right')
        
        plt.title(f'Top {top_features} Feature Coefficients in {model_name}')
        plt.tight_layout()
        plt.show()
        
        # Display coefficient table
        print("\nFeature Coefficient Ranking:")
        display(coef_df)
        
        # Feature correlation with target
        if hasattr(X, 'corrwith'):
            print("\nFeature Correlation with Target:")
            corr_df = pd.DataFrame({
                'Feature': feature_names,
                'Correlation': X.corrwith(pd.Series(y_values, index=X.index)).values
            })
            corr_df['Abs_Correlation'] = np.abs(corr_df['Correlation'])
            corr_df = corr_df.sort_values('Abs_Correlation', ascending=False)
            display(corr_df)
            
            # Plot correlation heatmap for top features
            plt.figure(figsize=(12, 10))
            top_features_list = coef_df.head(min(15, len(feature_names)))['Feature'].tolist()
            X_top = X[top_features_list]
            
            # Add target to the correlation matrix
            X_with_y = X_top.copy()
            X_with_y['Target'] = y_values
            
            # Plot correlation heatmap
            sns.heatmap(X_with_y.corr(), annot=True, cmap='coolwarm', vmin=-1, vmax=1)
            plt.title('Correlation Heatmap of Top Features with Target ({model_name})')
            plt.tight_layout()
            plt.show()

In [ ]:
from scipy.stats import norm
from scipy.optimize import curve_fit

def cdfR(U_model, r_eff, sigma_R, sigma_R_scale=1.0):
    # Calculate the argument for the CDF (the z-score)
    # Using log properties: log(a) + log(b) = log(a*b)
    log_argument = np.log(r_eff * U_model)
    z_score = log_argument / (sigma_R * sigma_R_scale)

    # Apply the standard normal cumulative distribution function (CDF)
    probability = norm.cdf(z_score)
    return probability

def ventilationModel(u_model, a, r_eff, sigma_r):
    return a * u_model * cdfR(u_model, r_eff, sigma_r)

def ventilationModelTrub(X, a, r_eff, sigma_R):
    U_model, sigma_R_scale = X
    return a * U_model * cdfR(U_model, r_eff, sigma_R, sigma_R_scale=sigma_R_scale)

def ventilationmodelCdConst(u_model, r_eff, sigma_r):
    return u_model * cdfR(u_model, r_eff, sigma_r)


In [ ]:
import scipy as sp
data = df.copy()
data[y_var] = ydf

adjustData = True
n_rows = 2 - int(adjustData)

fig, axs = plt.subplots(n_rows, 2, figsize=(12, 6*n_rows), dpi=140, sharex=True, sharey=True)
axs = axs.flatten()
for i in range(len(axs)):
    axs[i].grid(True, alpha=0.3)
    axs[i].tick_params(labelsize=14)
    axs[i].set_xlabel("Modeled Flux Velocity", fontsize=14)
    axs[i].set_ylabel("LES Flux Velocity", fontsize=14)

hue, style = "roomType", "slAll"

for i in range(2):
    sl_val = bool(i)
    plotdf = data[data["skylight"] == sl_val]
    plotdf = plotdf.sort_values(by=[hue, style]) # Sort for consistent coloring
    min_val = min(plotdf[x_var].min(), plotdf[y_var].min())
    max_val = max(plotdf[x_var].max(), plotdf[y_var].max())

    # 1:1 reference
    xs = np.array([min_val, max_val])
    axs[i].plot(xs, xs, 'r--', label='1:1 Line')
    # add ±25% reference lines
    # axs[i].plot(xs, xs * 1.25, color='gray', linestyle='--', label='+25%')
    # axs[i].plot(xs, xs * 0.75, color='gray', linestyle='--', label='-25%')

    # title and axes
    title = "Skylight" if sl_val else "Window"
    axs[i].set_title(title, fontsize=20)

    # two regression lines per subplot (Sdelp = +1, -1)
    for s, stl, lbl in [(1, '-', 'Flow Entering'), (-1, '-', 'Flow Exiting')]:
        regdf = plotdf[plotdf["Sdelp"] == s][[x_var, y_var]]
        regdf_abs = regdf.abs()
        regdf_abs = regdf_abs.sort_values(by=x_var)
        popt = curve_fit(
            ventilationModel,
            regdf_abs[x_var],
            regdf_abs[y_var],
            p0=[1.0, 0.5, 1.0],
            bounds=([0.1, 0, 0], [np.inf, np.inf, 2])
        )[0]
        print(f"Fitted parameters for {title}, {lbl}: popt={popt}")
        print(f"Cd = {0.6*popt[0]:.2f}, r_eff = {np.log(popt[1]):.2f}, sigma_R = {popt[2]:.2f}")
        probs = pd.Series(cdfR(regdf_abs[x_var], popt[1], popt[2]))
        probs_stats = probs.describe()
        # print(f"CDF Stats for {title}, 25%: {probs_stats['25%']:.4f}, 50%: {probs_stats['50%']:.4f}, 75%: {probs_stats['75%']:.4f}")
        
        if adjustData:
            # Adjust y values based on fitted model
            plotdf.loc[plotdf["Sdelp"] == s, x_var] = plotdf[plotdf["Sdelp"] == s][x_var].apply(lambda x: ventilationModel(np.abs(x), *popt) * s)
        else:
            # plot regression line
            y = ventilationModel(regdf_abs[x_var], *popt)
            x_plot = regdf_abs[x_var] * s
            y_plot = y * s

            # Fit line on both primary subplots (left and corresponding bottom)
            if s > 0:
                fit_label = "Fit Ventilation Model"
            else:
                fit_label = None
            axs[i].plot(x_plot, y_plot, color="k", linestyle=stl, label=fit_label, linewidth=2.5)
            axs[i+2].plot(x_plot, y_plot, color="k", linestyle=stl, label=fit_label, linewidth=2.5)
            # Probability on secondary y-axis of the bottom subplot
            ax2 = axs[i+2].twinx()
            ax2.plot(x_plot, probs, color="blue", linestyle=':', label='Ventilation CDF')
            ax2.set_ylabel('Probability', fontsize=14)
            ax2.set_ylim(0, 1.05)
            ax2.tick_params(axis='y', labelsize=12)
            ax2.grid(False)

            
            # Plot theoretical "Ventilation When On" reference on bottom subplot
            if s > 0:
                label = "Bousinesq Ventilation"
            else:
                label = None
            axs[i+2].plot(x_plot, regdf_abs[x_var] * s * popt[0],
                          color="k", linestyle='-.', label=label, linewidth=1.5)
            if i == 0:
                # Combine legends from primary and secondary axes, deduplicating by label
                h1, l1 = axs[i+2].get_legend_handles_labels()
                h2, l2 = ax2.get_legend_handles_labels()
                handles = h1 + h2
                labels = l1 + l2

                filtered = [(h, l) for h, l in zip(handles, labels) if l not in (None, '', '_nolegend_')]
                ax2.legend(
                    [h for h, _ in filtered],
                    [l for _, l in filtered],
                    loc='upper left',
                    fontsize=12
                )    
        rmse, nrmse  = calculate_normalized_rmse(plotdf.loc[plotdf["Sdelp"] == s, x_var], plotdf.loc[plotdf["Sdelp"] == s, y_var], normalization='std')
        print(f"NRMSE: {nrmse:.2f}, RMSE: {rmse:.3f}")
        error = plotdf.loc[plotdf["Sdelp"] == s, x_var] - plotdf.loc[plotdf["Sdelp"] == s, y_var]
        print(f"Bias: {np.mean(error):.4f}, Error STD: {np.std(error):.4f}")

    sns.scatterplot(
        data=plotdf, x=x_var, y=y_var,
        hue=plotdf[hue], 
        style=plotdf[style],
        alpha=0.6, s=50, ax=axs[i]
    )
    # rmse, nrmse  = calculate_normalized_rmse(plotdf[x_var], plotdf[y_var], normalization='std')
    # print(f"For in and out and skylights {sl_val}, NRMSE: {nrmse:.4f}, RMSE: {rmse:.4f}")

    # Set legend with custom labels
for i in range(len(axs)):
    if i == 0:  # Only add detailed legend to first subplot
        handles, labels = axs[i].get_legend_handles_labels()
        axs[i].legend(loc='upper left', fontsize=16, title='Room Type', title_fontsize='12',)
    elif axs[i].get_legend() is not None:
        axs[i].get_legend().remove()  # Remove redundant legends

fig.suptitle("Modeled vs LES Flux with Sign‐group Fits", fontsize=22)
plt.tight_layout(rect=[0,0,1,0.95])

# rmse, nrmse = calculate_normalized_rmse(data[y_var], data[x_var], normalization='std')
# print(f"Overall NRMSE: {nrmse:.4f}, RMSE: {rmse:.4f}")


In [ ]:
data = df.copy()
data[y_var] = ydf

fig, axs = plt.subplots(1, 2, figsize=(12, 6), dpi=140, sharex=True, sharey=True)
axs = axs.flatten()

hue, style = "roomType", "slAll"
x_var2 = "p_rms-noInt-Norm"
# x_var2 = "rms-mass_flux-Norm"
adjustData = True

for i in range(2):
    sl_val = bool(i)
    plotdf = data[data["skylight"] == sl_val]
    plotdf = plotdf.sort_values(by=[hue, style]) # Sort for consistent coloring
    min_val = min(plotdf[x_var].min(), plotdf[y_var].min())
    max_val = max(plotdf[x_var].max(), plotdf[y_var].max())

    # 1:1 reference
    xs = np.array([min_val, max_val])
    axs[i].plot(xs, xs, 'r--', label='1:1 Line')
    # add ±25% reference lines
    # axs[i].plot(xs, xs * 1.25, color='gray', linestyle='--', label='+25%')
    # axs[i].plot(xs, xs * 0.75, color='gray', linestyle='--', label='-25%')

    # title and axes
    title = "Skylight" if sl_val else "Window"
    axs[i].set_title(title, fontsize=20)
    axs[i].set_xlabel("Modeled Flux Velocity")
    axs[i].set_ylabel("LES Volume Flux Velocity")
    axs[i].grid(True, alpha=0.3)
    axs[i].tick_params(labelsize=14)

    # two regression lines per subplot (Sdelp = +1, -1)
    for s, stl, lbl in [(1, '-', 'Flow Entering'), (-1, '-.', 'Flow Exiting')]:
        regdf = plotdf[plotdf["Sdelp"] == s][[x_var, x_var2, y_var]]
        regdf_abs = regdf.abs()
        regdf_abs = regdf_abs.sort_values(by=x_var)
        popt = curve_fit(
            ventilationModelTrub,
            (regdf_abs[x_var], regdf_abs[x_var2]),
            regdf_abs[y_var],
            p0=[1.0, 0.5, 100.0],
            bounds=([0.1, 0, 0], [np.inf, np.inf, np.inf])
        )[0]
        print(f"Fitted parameters for {title}, {lbl}: popt={popt}")
        if adjustData:
            # Adjust y values based on fitted model
            mask = plotdf["Sdelp"] == s
            U = np.abs(plotdf.loc[mask, x_var].values)
            sigma_scale = np.abs(plotdf.loc[mask, x_var2].values)
            preds = ventilationModelTrub((U, sigma_scale), *popt) * s
            plotdf.loc[mask, x_var] = preds
        else:
            # plot regression line
            y = ventilationModelTrub((regdf_abs[x_var].values, regdf_abs[x_var2].values), *popt)
            axs[i].plot(regdf_abs[x_var]*s, y*s, color="k", linestyle=stl, label=lbl, linewidth=2.5)
            

    sns.scatterplot(
        data=plotdf, x=x_var, y=y_var,
        hue=plotdf[hue], 
        style=plotdf[style],
        alpha=0.6, s=50, ax=axs[i]
    )

    # Set legend with custom labels
    if i == 1:  # Only add detailed legend to first subplot
        handles, labels = axs[i].get_legend_handles_labels()
        axs[i].legend(loc='upper left', fontsize=16, title='Room Type', title_fontsize='18',)
    elif axs[i].get_legend() is not None:
        axs[i].get_legend().remove()  # Remove redundant legends
    axs[i].set_aspect('equal', adjustable='box')

    rmse, nrmse  = calculate_normalized_rmse(plotdf[x_var], plotdf[y_var], normalization='std')
    print(f"For {sl_val}, NRMSE: {nrmse:.4f}, RMSE: {rmse:.4f}")

fig.suptitle("Modeled vs LES Flux with Sign‐group Fits", fontsize=22)
plt.tight_layout(rect=[0,0,1,0.95])

